
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lab - Chunking PDFs and AI Search
#### Duration: 40 minutes
## Overview
In this lab, you will take a PDF through the **Parse → Prep Search → AI Search** pipeline on your own. You will use `ai_parse_document` to extract text, `ai_prep_search` to chunk it, persist the chunks to a Delta table with Change Data Feed enabled, and create an AI Search index. Finally, you will query the index using three different search modes with `AISearchClient()` from the Databricks AI Search SDK.

## Learning Objectives
By the end of this lab, you will be able to:
1. **Parse a PDF and chunk the text** using `ai_parse_document` and `ai_prep_search`
2. **Write chunked data to a Delta table** with Change Data Feed enabled
3. **Create an AI Search index** on the `vs_all_demo` endpoint
4. **Test the index** with similarity, hybrid, and full-text keyword search queries

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Beta Features</strong>
<p style="margin: 8px 0 0 0; color: #333;">This notebook uses Databricks Beta Features. While this notebook has been thoroughly tested, it's worth noting that full functionality is not guaranteed.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 6</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

Run the following cell to configure your working environment. This sets your `catalog_name` and `schema_name` variables and makes the shared lab assets available to you.

In [0]:
%run ./Includes/Classroom-Setup-07

### A1. Review the Lab Variables

As a part of the workspace setup, certain assets have been managed for you (volumes, PDF, etc.). You will read the PDF `airbnb_house_rules.pdf` from the shared catalog `dbacademy`, schema `rag_all`, and volume `vol_all`.

1. Run the next cell to define the source PDF path, target table names, embedding model, and AI Search endpoint/index names.

2. Confirm the printed output shows your own catalog and schema — you will write all lab objects into your personal schema.

**NOTE:** These variables are referenced throughout the lab. Do not modify them.

In [0]:
# Copied into your volume by the classroom setup (Academy labs: /Volumes/dbacademy/rag_all/vol_all/...)
HOUSE_RULES_PDF = f"/Volumes/{catalog_name}/{schema_name}/house_rules_vol/additional_listing_info/airbnb_house_rules.pdf"
CHUNKS_TABLE = f"{catalog_name}.{schema_name}.house_rules_chunks"

EMBEDDING_MODEL = "databricks-gte-large-en"
VS_ENDPOINT = "vs_all_demo"
VS_INDEX = f"{catalog_name}.{schema_name}.house_rules_index"

print(f"Source PDF:    {HOUSE_RULES_PDF}")
print(f"Chunks table:  {CHUNKS_TABLE}")
print(f"VS Index:      {VS_INDEX}")

## B. Parse the PDF

### B1. Extract Elements with `ai_parse_document`

The `ai_parse_document` function returns a `VARIANT` value describing the layout of a document: its pages, elements, and the text content of each element. To work with individual elements as rows, you explode the `elements` array out of the returned `VARIANT`.

Complete the following in the TODO cell below:

1. Call `ai_parse_document()` on the `content` column, passing `MAP('version', '2.0')` as the options argument.

2. Read the source PDF using `READ_FILES()` with the `binaryFile` format.

3. Use `variant_explode()` on the `elements` array to produce one row per document element.

4. Save the resulting DataFrame as a table named `parsed_house_rules`.

**NOTE:** This will parse ~300 PDF elements and may take a minute or two to complete.

In [0]:
## Parse the house rules PDF and flatten its elements into rows.
parsed_df = spark.sql(f"""
  WITH parsed AS (
    SELECT
      path,
      <FILL_IN>(content, <FILL_IN>) AS parsed
    FROM READ_FILES('{HOUSE_RULES_PDF}', format => '<FILL_IN>')
  )
  SELECT
    path,
    element.value:type::STRING AS element_type,
    element.value:content::STRING AS content
  FROM parsed,
    LATERAL <FILL_IN>(parsed.parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
""")

parsed_df.write.mode("overwrite").saveAsTable(<FILL_IN>)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
parsed_df = spark.sql(f"""
  WITH parsed AS (
    SELECT
      path,
      ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{HOUSE_RULES_PDF}', format => 'binaryFile')
  )
  SELECT
    path,
    element.value:type::STRING AS element_type,
    element.value:content::STRING AS content
  FROM parsed,
    LATERAL variant_explode(parsed.parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
""")

parsed_df.write.mode("overwrite").saveAsTable("parsed_house_rules")
```

</div>
</details>

### B2. Inspect the Parsed Elements

1. Complete the SQL query below to select all rows from the `parsed_house_rules` table.

2. Review the `element_type` column values. Notice the variety of element types (text, section headers, tables, etc.) that `ai_parse_document` identifies.

In [0]:
%sql
---- Select all rows from the parsed elements table
SELECT <FILL_IN> FROM <FILL_IN>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```sql
SELECT * FROM parsed_house_rules
```

</div>
</details>

## C. Chunk with `ai_prep_search`

### C1. Generate Chunks

The `ai_prep_search` function intelligently chunks parsed text. It uses AI models to split text at semantically meaningful boundaries, which produces higher-quality retrieval units than simple fixed-size splitting.

**IMPORTANT:** `ai_prep_search()` needs the full `ai_parse_document` `VARIANT` per document — not the flattened elements stored in `parsed_house_rules`. This means you must run it against the original parsed output rather than the table you created in Section B.

Complete the following in the TODO cell below:

1. Pass the full `parsed` `VARIANT` column into `ai_prep_search()`.

2. Explode `prepped:document:contents` to produce one row per chunk.

3. Extract the `chunk_id`, `chunk_position`, `chunk_to_retrieve`, and `chunk_to_embed` fields from each chunk.

4. Save the results to a table named `house_rules_chunks`, enabling `mergeSchema`.

In [0]:
## Chunk the parsed document using ai_prep_search and save the results.
chunks_df = spark.sql(f"""
  WITH parsed AS (
    SELECT
      path,
      ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{HOUSE_RULES_PDF}', format => 'binaryFile')
  ),
  prepped AS (
    SELECT
      path,
      <FILL_IN>(parsed) AS prepped
    FROM parsed
  )
  SELECT
    p.path,
    chunk.value:chunk_id::STRING        AS chunk_id,
    chunk.value:chunk_position::INT     AS chunk_position,
    chunk.value:<FILL_IN>::STRING       AS chunk_to_retrieve,
    chunk.value:<FILL_IN>::STRING       AS chunk_to_embed
  FROM prepped p,
       LATERAL variant_explode(p.prepped:document:<FILL_IN>) AS chunk
""")

chunks_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(<FILL_IN>)


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
chunks_df = spark.sql(f"""
  WITH parsed AS (
    SELECT
      path,
      ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{HOUSE_RULES_PDF}', format => 'binaryFile')
  ),
  prepped AS (
    SELECT
      path,
      ai_prep_search(parsed) AS prepped
    FROM parsed
  )
  SELECT
    p.path,
    chunk.value:chunk_id::STRING        AS chunk_id,
    chunk.value:chunk_position::INT     AS chunk_position,
    chunk.value:chunk_to_retrieve::STRING AS chunk_to_retrieve,
    chunk.value:chunk_to_embed::STRING  AS chunk_to_embed
  FROM prepped p,
       LATERAL variant_explode(p.prepped:document:contents) AS chunk
""")

chunks_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("house_rules_chunks")
```

</div>
</details>

### C2. Understanding the Output

The chunks table contains two columns of particular interest as you build an AI Search index:

| Column | Purpose | Description |
|---|---|---|
| `chunk_to_retrieve` | Shown to the LLM after retrieval | The raw (or near-raw) chunk text, preserving full content and structure so the model has everything it needs to answer questions. |
| `chunk_to_embed` | Sent to the embedding model | A cleaned, context-enriched string that prepends document-level metadata (title, section headers, page number, captions, etc.) to the chunk text, optimized purely for retrieval quality. |

Complete the following:

1. Fill in the query below to select all rows from `house_rules_chunks`.

2. In the first row of the produced table, click the dropdown to expand the `chunk_to_retrieve` and `chunk_to_embed` columns.

3. Inspect the results and confirm that the raw text is stored in `chunk_to_retrieve` while `chunk_to_embed` holds the context-enriched text prepared for embeddings. To see the format of `chunk_to_embed`, review the documentation ([AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_prep_search#chunk_to_embed-format) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/ai_prep_search#chunk_to_embed-format) | [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/ai_prep_search#chunk_to_embed-format)).

In [0]:
%sql
---- Select all rows from the chunks table
SELECT <FILL_IN> FROM <FILL_IN>


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```sql
SELECT * FROM house_rules_chunks
```

</div>
</details>

## D. Write Chunks to Unity Catalog

You need to persist the chunks in a Delta table with Change Data Feed (CDF) enabled — this is **required** for AI Search to sync automatically. CDF tracks and records all changes (inserts, updates, deletes) to the table, allowing downstream systems to efficiently detect and process data modifications.

Complete the following in the TODO cell below:

1. Add a `chunk_id` column generated with the `uuid()` expression to serve as the primary key for the table and index.

2. Alias `chunk_to_embed` as `chunk_text` (the column AI Search will embed) and `chunk_to_retrieve` as `retrieval_text`.

3. Write the DataFrame to `CHUNKS_TABLE` in `delta` format using `overwrite` mode with `overwriteSchema` enabled.

4. Set the `delta.enableChangeDataFeed` table property to `true`.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<strong style="color: #0d47a1;">Learn more about CDF</strong>
<p style="margin: 8px 0 0 0; color: #333;">
See the Databricks documentation on Change Data Feed (<a href="https://docs.databricks.com/aws/en/delta/delta-change-data-feed.html" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/tables/features/change-data-feed" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/delta/delta-change-data-feed.html" style="color: #1976d2; text-decoration: underline;">GCP</a>) for a deeper dive.
</p>
</div>

In [0]:
## Map ai_prep_search outputs into your chunks table schema and write with CDF enabled.
from pyspark.sql.functions import expr, lit, col

chunks_with_id = (
    chunks_df
    .withColumn("chunk_id", expr("<FILL_IN>"))            ## PK for the table / index
    .withColumn("source_path", col("path"))
    .select(
        "chunk_id",
        col("<FILL_IN>").alias("chunk_text"),             ## for embeddings
        col("<FILL_IN>").alias("retrieval_text"),
        "source_path"
    )
)

chunks_with_id.write.format("delta") \
    .mode("<FILL_IN>") \
    .option("overwriteSchema", "true") \
    .option("<FILL_IN>", "true") \
    .saveAsTable(<FILL_IN>)

spark.sql(f"ALTER TABLE {CHUNKS_TABLE} SET TBLPROPERTIES (<FILL_IN> = true)")

print(f"Chunks table written: {CHUNKS_TABLE}")
display(spark.table(CHUNKS_TABLE))


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
from pyspark.sql.functions import expr, lit, col

# Map ai_prep_search outputs into your chunks table schema
chunks_with_id = (
    chunks_df
    .withColumn("chunk_id", expr("uuid()"))            # PK for the table / index
    .withColumn("source_path", col("path"))
    .select(
        "chunk_id",
        col("chunk_to_embed").alias("chunk_text"),     # for embeddings
        col("chunk_to_retrieve").alias("retrieval_text"),
        "source_path"
    )
)

chunks_with_id.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(CHUNKS_TABLE)

spark.sql(f"ALTER TABLE {CHUNKS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"Chunks table written: {CHUNKS_TABLE}")
display(spark.table(CHUNKS_TABLE))
```

</div>
</details>

## E. Create AI Search Index

### E1. Verify the Endpoint Is Online

You will attach an AI Search index to the shared `vs_all_demo` endpoint. Before creating the index, confirm the endpoint is in an `ONLINE` state.

1. Run the next cell to poll the endpoint status until it reports `ONLINE`.

2. You can also validate the endpoint manually by navigating to **Compute > AI Search** and viewing the **Status** column.

**NOTE:** As part of the workspace setup, the `vs_all_demo` endpoint should already be ready by the time you reach this point in the lab. This cell is provided for you — no changes needed.

In [0]:
from databricks.ai_search.client import AISearchClient
import time

client = AISearchClient()

## Verify the endpoint is ready
for _ in range(150):
    try:
        ep = client.get_endpoint(VS_ENDPOINT)
        status = ep.get("endpoint_status", {}).get("state", "")
        if status == "ONLINE":
            print(f"Endpoint '{VS_ENDPOINT}' is ONLINE.")
            break
        print(f"  Endpoint status: {status} - waiting...")
        time.sleep(10)
    except Exception:
        print(f"  Endpoint '{VS_ENDPOINT}' not found yet - waiting...")
        time.sleep(10)
else:
    print("WARNING: Endpoint not ready. The index creation may fail.")

### E2. Create the Delta Sync Index

Create a **Delta Sync Index** with **managed embeddings**, meaning Databricks automatically computes embeddings from the source column you designate.

Complete the following in the TODO cell below:

1. Call `create_delta_sync_index()` on the `AISearchClient` object `client`.

2. Set `source_table_name` to `CHUNKS_TABLE` and `index_name` to `VS_INDEX`.

3. Set `primary_key` to the column that uniquely identifies each chunk.

4. Set `embedding_source_column` to the column holding the context-enriched text and `embedding_model_endpoint_name` to `EMBEDDING_MODEL`.

5. Use `pipeline_type="TRIGGERED"` so the index syncs on demand rather than continuously.

**NOTE:** Index creation and the initial sync can take several minutes. The `except` block re-syncs the index if it already exists, so re-running the cell is safe.

In [0]:
## Create the Delta Sync index if it doesn't already exist
try:
    index = client.<FILL_IN>(
        endpoint_name=VS_ENDPOINT,
        source_table_name=<FILL_IN>,
        index_name=<FILL_IN>,
        pipeline_type="<FILL_IN>",
        primary_key="<FILL_IN>",
        embedding_source_column="<FILL_IN>",
        embedding_model_endpoint_name=<FILL_IN>,
        columns_to_sync=["chunk_id", "chunk_text", "source_path"],
    )
    print(f"Index '{VS_INDEX}' created and syncing.")
except Exception as e:
    if "already exists" in str(e):
        index = client.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
        print(f"Index '{VS_INDEX}' already exists. Syncing latest data.")
        index.sync()
    else:
        raise

print(index.describe())


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
# Create the index if it doesn't already exist
try:
    index = client.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        source_table_name=CHUNKS_TABLE,
        index_name=VS_INDEX,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=EMBEDDING_MODEL,
        columns_to_sync=["chunk_id", "chunk_text", "source_path"],
    )
    print(f"Index '{VS_INDEX}' created and syncing.")
except Exception as e:
    if "already exists" in str(e):
        index = client.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
        print(f"Index '{VS_INDEX}' already exists. Syncing latest data.")
        index.sync()
    else:
        raise

print(index.describe())
```

</div>
</details>

## F. Test the Index

Now that the index is online, test it with three different search modes:

| Subsection | Search Mode | `query_type` | How It Works |
|---|---|---|---|
| F1 | Similarity Search (Vector ANN) | *default* | Semantic vector similarity using Approximate Nearest Neighbor |
| F2 | Hybrid Search | `hybrid` | Combines vector and BM25 keyword matching via Reciprocal Rank Fusion (RRF) |
| F3 | Full-Text Keyword Search (Beta) | `FULL_TEXT` | BM25 keyword-only matching, no embeddings |

### F1. Similarity Search (Vector ANN)

The default search mode uses Approximate Nearest Neighbor (ANN) vector similarity. The index converts your query text into an embedding and finds the most semantically similar chunks.

Complete the following in the TODO cell below:

1. Retrieve the index with `client.get_index()` using `VS_ENDPOINT` and `VS_INDEX`, then print its detailed state.

2. Call `similarity_search()` with the query text *"What are the house rules about pets?"*.

3. Return the `chunk_id` and `chunk_text` columns, limited to 3 results.

**NOTE:** If the index is still syncing, wait a few minutes and re-run the cell.

In [0]:
## Retrieve the index and run a default vector similarity search
index = client.<FILL_IN>(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
status = index.describe().get("status", {}).get("detailed_state")
print(f"Index status: {status}")

results = index.<FILL_IN>(
    query_text="<FILL_IN>",
    columns=[<FILL_IN>],
    num_results=<FILL_IN>,
)
print(f"\n--- F1: Similarity Search (Vector ANN) ---")
for row in results.get("result", {}).get("data_array", []):
    print(f"\n--- Chunk ---")
    print(row[1])


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
index = client.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
status = index.describe().get("status", {}).get("detailed_state")
print(f"Index status: {status}")

results = index.similarity_search(
    query_text="What are the house rules about pets?",
    columns=["chunk_id", "chunk_text"],
    num_results=3,
)
print(f"\n--- F1: Similarity Search (Vector ANN) ---")
for row in results.get("result", {}).get("data_array", []):
    print(f"\n--- Chunk ---")
    print(row[1])
```

</div>
</details>

### F2. Hybrid Search (ANN + Keyword)

Hybrid search combines vector similarity (ANN) with BM25 full-text keyword matching, merging results via Reciprocal Rank Fusion (RRF). This mode is useful when queries contain exact terms that must match — such as specific names, error codes, or technical terminology — where pure semantic search might miss keyword-specific results.

1. Complete the TODO cell below by setting the `query_type` argument to perform a hybrid search.

2. Use the same query text and result count as F1 so you can compare the returned chunks directly.

In [0]:
## Run a hybrid (ANN + BM25) search using the same query
results_hybrid = index.similarity_search(
    query_text="What are the house rules about pets?",
    columns=["chunk_id", "chunk_text"],
    num_results=3,
    query_type="<FILL_IN>",
)
print(f"\n--- F2: Hybrid Search (ANN + Keyword, RRF) ---")
for row in results_hybrid.get("result", {}).get("data_array", []):
    print(f"\n--- Chunk ---")
    print(row[1])


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
results_hybrid = index.similarity_search(
    query_text="What are the house rules about pets?",
    columns=["chunk_id", "chunk_text"],
    num_results=3,
    query_type="hybrid",
)
print(f"\n--- F2: Hybrid Search (ANN + Keyword, RRF) ---")
for row in results_hybrid.get("result", {}).get("data_array", []):
    print(f"\n--- Chunk ---")
    print(row[1])
```

</div>
</details>

### F3. Full-Text Keyword Search (Beta)

Full-text keyword search uses only BM25 scoring — no vector embeddings are involved. This is useful when you need exact keyword matches without semantic interpretation.

1. Complete the TODO cell below by setting the `query_type` argument to perform a keyword-only search.

2. Compare the returned chunks against the F1 and F2 results. Which mode surfaced the most relevant chunk for this query?

**NOTE:** This search mode is currently in Beta.

In [0]:
## Run a BM25 keyword-only search using the same query
results_keyword = index.similarity_search(
    query_text="What are the house rules about pets?",
    columns=["chunk_id", "chunk_text"],
    num_results=3,
    query_type="<FILL_IN>",
)
print(f"\n--- F3: Full-Text Keyword Search ---")
for row in results_keyword.get("result", {}).get("data_array", []):
    print(f"\n--- Chunk ---")
    print(row[1])


<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #2574B5); color: white; padding: 14px 20px; cursor: pointer; font-weight: 700; font-size: 13pt; border-radius: 8px; user-select: none;">
Click to view answer
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 18px 20px; background: #F8F9FC; position: relative;"><button onclick="var c=this.parentElement.querySelector('pre code');var t=document.createElement('textarea');t.value=c?c.textContent:'';t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();document.execCommand('copy');document.body.removeChild(t);this.textContent='Copied!';var b=this;setTimeout(function(){b.textContent='Copy'},1500)" style="position:absolute;top:10px;right:12px;background:linear-gradient(135deg,#1B5162,#2574B5);color:#fff;border:none;border-radius:6px;padding:6px 14px;font-size:11pt;font-weight:600;cursor:pointer;z-index:2;">Copy</button>

```python
results_keyword = index.similarity_search(
    query_text="What are the house rules about pets?",
    columns=["chunk_id", "chunk_text"],
    num_results=3,
    query_type="FULL_TEXT",
)
print(f"\n--- F3: Full-Text Keyword Search ---")
for row in results_keyword.get("result", {}).get("data_array", []):
    print(f"\n--- Chunk ---")
    print(row[1])
```

</div>
</details>

## G. Conclusion

In this lab, you:

1. **Parsed** a PDF with `ai_parse_document` and flattened its elements into rows
2. **Chunked** the text with `ai_prep_search` for semantically meaningful splits, distinguishing `chunk_to_retrieve` from `chunk_to_embed`
3. **Stored** chunks in a Delta table with Change Data Feed enabled
4. **Created** an AI Search Delta Sync index with managed embeddings on the shared `vs_all_demo` endpoint
5. **Tested** the index using similarity, hybrid, and full-text keyword search

### Next Steps

With a working retrieval index in place, you are ready to wire it into a retrieval-augmented generation (RAG) application. Consider how the choice of search mode and the distinction between embedded and retrieved text will affect the quality of the context you pass to the LLM.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>